In [ ]:
# build_features.py
# Modified to preserve 'id' column for submission alignment

import os
import numpy as np
import pandas as pd
import pickle

PATH = "/kaggle/input/store-sales-time-series-forecasting"

# ---------------------------
# 1) LOAD
# ---------------------------
train = pd.read_csv(os.path.join(PATH, "train.csv"), parse_dates=["date"])
test  = pd.read_csv(os.path.join(PATH, "test.csv"),  parse_dates=["date"])
oil   = pd.read_csv(os.path.join(PATH, "oil.csv"), parse_dates=["date"]).rename(columns={"dcoilwtico": "oil"})
stores = pd.read_csv(os.path.join(PATH, "stores.csv"))
transactions = pd.read_csv(os.path.join(PATH, "transactions.csv"), parse_dates=["date"])
holidays = pd.read_csv(os.path.join(PATH, "holidays_events.csv"), parse_dates=["date"])

# train/test windows
train_start = train.date.min().normalize()
train_end   = train.date.max().normalize()
test_start  = test.date.min().normalize()
test_end    = test.date.max().normalize()

# ---------------------------
# 2) Reindex training to full grid (date, store, family)
# ---------------------------
full_idx = pd.MultiIndex.from_product(
    [pd.date_range(train_start, train_end), train.store_nbr.unique(), train.family.unique()],
    names=["date", "store_nbr", "family"],
)
train = (
    train.set_index(["date", "store_nbr", "family"])
         .reindex(full_idx)
         .reset_index()
         .sort_values(["date", "store_nbr", "family"], ignore_index=True)
)
train[["sales", "onpromotion"]] = train[["sales", "onpromotion"]].fillna(0.0)

# NOTE: train data doesn't have 'id', only test does - we'll add it later if needed

# ---------------------------
# 3) External series cleanup
# ---------------------------
# Oil: fill daily, interpolate safely
oil = oil.merge(pd.DataFrame({"date": pd.date_range(train_start, test_end)}), on="date", how="outer")
oil = oil.sort_values("date", ignore_index=True)
oil["oil"] = oil["oil"].interpolate(method="linear", limit_direction="both")

# Transactions: fill missing days and interpolate per-store
store_sales = train.groupby(["date", "store_nbr"]).sales.sum().reset_index()

# Create full grid of dates and stores
date_range = pd.date_range(train_start, train_end)
store_numbers = stores["store_nbr"].unique()
full_grid = pd.MultiIndex.from_product(
    [date_range, store_numbers], 
    names=["date", "store_nbr"]
).to_frame(index=False)

# Merge original transactions into the full grid
transactions = full_grid.merge(transactions, on=["date", "store_nbr"], how="left")

# Merge store sales to help with interpolation logic
transactions = transactions.merge(store_sales, on=["date", "store_nbr"], how="left")

# Convert transactions to float
transactions["transactions"] = transactions["transactions"].astype(float)

# If sales is 0, set transactions to 0
transactions.loc[transactions["sales"].eq(0) & transactions["transactions"].isna(), "transactions"] = 0.0

# Interpolate per store
transactions["transactions"] = transactions.groupby("store_nbr", group_keys=False)["transactions"].apply(
    lambda s: s.interpolate(method="linear", limit_direction="both")
)

# Drop sales column and sort
transactions = transactions.drop(columns=["sales"]).sort_values(["date", "store_nbr"], ignore_index=True)

# ---------------------------
# 4) Holidays — transfer handling + useful flags
# ---------------------------
hol = holidays.copy()

# Resolve transfers: build a "celebrated date" field
hol["is_transfer"] = hol["type"].eq("Transfer")
transfer_map = hol.loc[hol["is_transfer"], ["date", "description"]].rename(
    columns={"date": "celebrated_date"}
)
hol = hol.merge(transfer_map, on="description", how="left")
hol["celebrated_date"] = hol["celebrated_date"].fillna(hol["date"])
hol = hol[~hol["is_transfer"]].copy()

# Bridge & Work Day retained as-is
hol["is_bridge"] = hol["type"].eq("Bridge").astype(int)
hol["is_workday"] = hol["type"].eq("Work Day").astype(int)

# Normalize description for national scope
hol["desc_norm"] = hol["description"].str.lower().str.replace(r"[-_]", " ", regex=True).str.strip()
nat = hol[hol["locale"] == "National"][["celebrated_date", "desc_norm"]].rename(columns={"celebrated_date": "date"})
nat = nat.drop_duplicates().sort_values("date", ignore_index=True)

# One-hot select common national holidays
topN = 12
top_desc = nat["desc_norm"].value_counts().head(topN).index.tolist()
nat["desc_norm"] = nat["desc_norm"].where(nat["desc_norm"].isin(top_desc), "other_nat_holiday")
nat_oh = pd.get_dummies(nat, columns=["desc_norm"], prefix="nat").groupby("date").sum().reset_index()

# ---------------------------
# 5) Combine everything at the row level
# ---------------------------
def add_payday_flags(df):
    df["is_15th"] = (df.date.dt.day == 15).astype(int)
    next_day = df["date"] + pd.Timedelta(days=1)
    df["is_eom"] = (next_day.dt.day == 1).astype(int)
    df["is_q_start"] = df["date"].dt.is_quarter_start.astype(int)
    df["is_q_end"]   = df["date"].dt.is_quarter_end.astype(int)
    return df

# Build a unified frame (train+test for feature creation)
data = pd.concat(
    [train, test], axis=0, ignore_index=True
).merge(
    oil, on="date", how="left"
).merge(
    stores, on="store_nbr", how="left"
).merge(
    transactions, on=["date", "store_nbr"], how="left"
).merge(
    nat_oh, on="date", how="left"
).sort_values(["store_nbr", "family", "date"], ignore_index=True)

# Fill holiday one-hots with 0
nat_cols = [c for c in data.columns if c.startswith("nat_")]
for c in nat_cols:
    data[c] = data[c].fillna(0).astype(int)

# Work day flag
work_days = hol[hol["is_workday"] == 1][["celebrated_date"]].rename(columns={"celebrated_date": "date"})
work_days["work_day"] = 1
data = data.merge(work_days, on="date", how="left")
data["work_day"] = data["work_day"].fillna(0).astype(int)

# Time features
data["day"] = data.date.dt.day
data["month"] = data.date.dt.month
data["year"] = data.date.dt.year
data["day_of_week"] = data.date.dt.dayofweek
data["week_of_year"] = data.date.dt.isocalendar().week.astype(int)
data["day_of_year"] = data.date.dt.dayofyear
data["is_weekend"] = (data.day_of_week >= 5).astype(int)
data["is_month_start"] = data.date.dt.is_month_start.astype(int)
data["is_month_end"] = data.date.dt.is_month_end.astype(int)

# Cyclical encodings
for col, period in [("day_of_week", 7), ("month", 12)]:
    data[f"{col}_sin"] = np.sin(2 * np.pi * data[col] / period)
    data[f"{col}_cos"] = np.cos(2 * np.pi * data[col] / period)

# Payday & quarter flags
data = add_payday_flags(data)

# ---------------------------
# 6) Lags, rollings, EWM
# ---------------------------
def by_series(df):
    return ["store_nbr", "family"]

def make_lags(df, col, lags, keys):
    for L in lags:
        df[f"{col}_lag_{L}"] = df.groupby(keys, group_keys=False)[col].shift(L)
    return df

def make_rollings(df, col, windows, keys):
    for w in windows:
        df[f"{col}_rm_{w}"] = df.groupby(keys, group_keys=False)[col].transform(lambda s: s.rolling(w, min_periods=1).mean())
        df[f"{col}_rs_{w}"] = df.groupby(keys, group_keys=False)[col].transform(lambda s: s.rolling(w, min_periods=1).std())
    return df

def make_ewm(df, col, alphas, keys):
    for a in alphas:
        df[f"{col}_ewm_{str(a).replace('.','_')}"] = df.groupby(keys, group_keys=False)[col].transform(lambda s: s.ewm(alpha=a, adjust=False).mean())
    return df

# sales-based histories
sales_lags = [1, 7, 14, 21, 28, 56, 182, 365]
data = make_lags(data, "sales", sales_lags, by_series(data))
data = make_rollings(data, "sales", [7, 14, 28], by_series(data))
data = make_ewm(data, "sales", [0.3], by_series(data))

# transactions lags
trans_lags = [16, 17, 18, 19, 20, 21, 22]
for L in trans_lags:
    data[f"transactions_lag_{L}"] = data.groupby("store_nbr", group_keys=False)["transactions"].shift(L)

# onpromotion moving averages
for w in [7, 28]:
    data[f"onpromotion_ma{w}"] = data.groupby(by_series(data), group_keys=False)["onpromotion"].transform(
        lambda s: s.rolling(w, min_periods=1).mean()
    )

# oil moving averages
for w in [7, 28]:
    data[f"oil_ma{w}"] = data["oil"].rolling(w, min_periods=1).mean()

# Fill NA in histories
hist_cols = [c for c in data.columns if any(p in c for p in ["sales_lag_", "sales_rm_", "sales_rs_", "sales_ewm_", "transactions_lag_", "onpromotion_ma", "oil_ma"])]
for c in hist_cols:
    if c.startswith("transactions"):
        data[c] = data.groupby("store_nbr")[c].ffill().fillna(0.0)
    elif "oil" in c:
        data[c] = data[c].ffill().fillna(method="bfill")
    else:
        data[c] = data.groupby(by_series(data))[c].ffill().fillna(0.0)

# ---------------------------
# 7) Categorical encoders
# ---------------------------
from sklearn.preprocessing import LabelEncoder

cat_cols_static = ["store_nbr", "family", "city", "state", "type", "cluster"]
cat_cols_dyn    = ["day_of_week", "month"]
cat_encoders = {}
for col in cat_cols_static + cat_cols_dyn:
    le = LabelEncoder()
    data[f"{col}_enc"] = le.fit_transform(data[col].astype(str))
    cat_encoders[col] = le

# ---------------------------
# 8) Feature groups (spec)
# ---------------------------
dynamic_future_cont = ["onpromotion", "oil", "work_day"] + nat_cols + [
    "day_of_week_sin", "day_of_week_cos", "month_sin", "month_cos",
    "is_15th", "is_eom", "is_q_start", "is_q_end",
    "is_month_start", "is_month_end", "is_weekend"
]
dynamic_future_cat  = ["day_of_week_enc", "month_enc"]

dynamic_past_cont = hist_cols + [
    "onpromotion", "transactions", "oil",
    "day_of_week_sin", "day_of_week_cos", "month_sin", "month_cos",
    "is_15th", "is_eom", "is_q_start", "is_q_end",
    "is_month_start", "is_month_end", "is_weekend", "work_day"
] + nat_cols

dynamic_past_cat = ["day_of_week_enc", "month_enc"]
static_cat = [f"{c}_enc" for c in cat_cols_static]

log1p_cont = (
    [c for c in dynamic_past_cont if c.startswith(("sales_", "transactions_", "onpromotion_ma"))] +
    [c for c in dynamic_future_cont if c in ["onpromotion", "transactions"]] +
    ["onpromotion", "transactions"]
)

feature_spec = {
    "static_categorical": static_cat,
    "dynamic_past_categorical": dynamic_past_cat,
    "dynamic_future_categorical": dynamic_future_cat,
    "dynamic_past_continuous": dynamic_past_cont,
    "dynamic_future_continuous": dynamic_future_cont,
    "log1p_continuous": sorted(set(log1p_cont)),
    "target": "sales",
    "notes": "Targets unscaled; model trains in log space. ID column preserved in test set for submission alignment."
}

# ---------------------------
# 9) Persist processed frames & spec
# ---------------------------
data = data.sort_values(["date", "store_nbr", "family"], ignore_index=True)

train_df = data[data["date"] <= train_end].copy()
test_df  = data[data["date"] >  train_end].copy()

# Ensure 'id' column exists in test_df
if "id" not in test_df.columns:
    print("WARNING: 'id' column not found in test data. This may cause submission alignment issues.")
else:
    print(f"✓ 'id' column preserved in test data ({test_df['id'].notna().sum()} non-null values)")

# Persist
train_df.to_pickle("processed_train_data_enhanced.pkl")
test_df.to_pickle("processed_test_data_enhanced.pkl")

with open("feature_spec_enhanced.pkl", "wb") as f:
    pickle.dump(feature_spec, f)

with open("cat_encoders_enhanced.pkl", "wb") as f:
    pickle.dump(cat_encoders, f)

# Metadata
metadata = {
    "train_start": train_start,
    "train_end": train_end,
    "test_start": test_start,
    "test_end": test_end,
    "nat_cols": nat_cols,
    "n_series": train_df.groupby(["store_nbr", "family"]).ngroups,
    "has_test_ids": "id" in test_df.columns,
}
with open("metadata_enhanced.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("✓ Features built")
print(f"Train rows: {len(train_df):,}  Test rows: {len(test_df):,}")
print(f"Series: {metadata['n_series']}")
print(f"Future-known cont: {len(dynamic_future_cont)}  Future cat: {len(dynamic_future_cat)}")
print(f"Past cont: {len(dynamic_past_cont)}  Past cat: {len(dynamic_past_cat)}")